<a href="https://colab.research.google.com/github/RudolfVonStroheim/RaceClassifier/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pandas-plink
!curl -O http://www.russiangenome.ru/biengi.zip
!unzip biengi.zip -d biengi
!wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20200121.zip -O plink.zip
!unzip -q -n plink.zip plink

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 33.0M  100 33.0M    0     0  7160k      0  0:00:04  0:00:04 --:--:-- 7161k
Archive:  biengi.zip
replace biengi/biengi.bed? [y]es, [n]o, [A]ll, [N]one, [r]ename: Т
error:  invalid response [Т]
replace biengi/biengi.bed? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
--2026-05-31 10:59:49--  http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20200121.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.212.67, 16.15.183.67, 16.182.96.208, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.212.67|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8916292 (8.5M) [application/zip]
Saving to: ‘plink.zip’

plink.zip           100%[===================>]   8.50M  10.9MB/s    in 0.8s    

2026-05-31 10:59:50 (10.9 MB/s) - ‘plink.zip’ saved [8916292/8916292]



In [10]:
!mkdir pruned
!./plink --bfile biengi/biengi --indep 50 5 2 --out biengi/biengi_pruned # Прунинг датасета(мажно сделать средствами python, но так занимает меньше времени)
!./plink --bfile biengi/biengi --extract biengi/biengi_pruned.prune.in --make-bed --out pruned/data_pruned

mkdir: cannot create directory ‘pruned’: File exists
PLINK v1.90b6.15 64-bit (21 Jan 2020)          www.cog-genomics.org/plink/1.9/
(C) 2005-2020 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to biengi/biengi_pruned.log.
Options in effect:
  --bfile biengi/biengi
  --indep 50 5 2
  --out biengi/biengi_pruned

12975 MB RAM detected; reserving 6487 MB for main workspace.
242180 variants loaded from .bim file.
894 people (574 males, 320 females) loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 894 founders and 0 nonfounders present.
Calculating allele frequencies... 0%1%2%3%4%5%6%7%8%9%10%11%12%13%14%15%16%17%18%19%20%21%22%23%24%25%26%27%28%29%30%31%32%33%34%35%36%37%38%39%40%41%42%43%44%45%46%47%48%49%50%51%52%53%54%55%56%57%58%59%60%61%

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas_plink import read_plink
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

In [20]:
# До прунинга
bim, fam, G = read_plink("biengi/biengi")
gn = pd.DataFrame(G.compute().T)
gn

Mapping files: 100%|██████████| 3/3 [00:00<00:00,  4.16it/s]


,0,1,2,3,4,5,6,7,8,9,...,242170,242171,242172,242173,242174,242175,242176,242177,242178,242179
0,1.0,2.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,...,2.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,2.0,2.0
1,1.0,2.0,2.0,1.0,2.0,2.0,2.0,0.0,2.0,1.0,...,1.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0
2,2.0,1.0,1.0,2.0,2.0,1.0,2.0,1.0,2.0,1.0,...,2.0,1.0,1.0,1.0,1.0,2.0,0.0,2.0,2.0,2.0
3,2.0,2.0,2.0,1.0,2.0,2.0,2.0,1.0,2.0,1.0,...,0.0,2.0,2.0,2.0,2.0,2.0,1.0,0.0,1.0,1.0
4,1.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,1.0,...,2.0,0.0,0.0,1.0,1.0,2.0,1.0,1.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,1.0,2.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,1.0,...,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
890,2.0,1.0,2.0,2.0,2.0,2.0,2.0,1.0,1.0,0.0,...,2.0,1.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0
891,2.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,2.0,...,2.0,2.0,2.0,1.0,0.0,2.0,1.0,2.0,2.0,2.0
892,2.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,0.0,...,1.0,1.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0


In [19]:
#После прунинга
bim, fam, G = read_plink("pruned/data_pruned")
gn_p = pd.DataFrame(G.compute().T)
gn_p

Mapping files: 100%|██████████| 3/3 [00:00<00:00,  7.04it/s]


,0,1,2,3,4,5,6,7,8,9,...,98592,98593,98594,98595,98596,98597,98598,98599,98600,98601
0,1.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0,0.0,2.0,...,2.0,1.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0
1,1.0,2.0,1.0,2.0,0.0,2.0,1.0,2.0,1.0,1.0,...,1.0,0.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,2.0
2,2.0,1.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,...,0.0,1.0,2.0,2.0,1.0,1.0,2.0,0.0,2.0,2.0
3,2.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,...,1.0,0.0,2.0,0.0,2.0,2.0,2.0,1.0,0.0,1.0
4,1.0,2.0,1.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,...,1.0,1.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,1.0,2.0,2.0,0.0,1.0,1.0,1.0,2.0,1.0,1.0,...,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
890,2.0,1.0,2.0,2.0,1.0,1.0,0.0,2.0,0.0,2.0,...,0.0,1.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0
891,2.0,1.0,2.0,1.0,1.0,1.0,2.0,2.0,1.0,2.0,...,2.0,2.0,2.0,2.0,1.0,0.0,2.0,1.0,2.0,2.0
892,2.0,1.0,2.0,2.0,2.0,2.0,0.0,2.0,0.0,2.0,...,2.0,0.0,2.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0


В результате LD-прунинга количество признаков удалось существенно сократить.
Необходимость данного метода обуславливается спецификой данных: в геноме большое количество блоков очень часто наследуются вместе и при обработке необходимо учитывать весь блок целиком, а не его фрагменты. В итоге число признаков сократилось чуть более, чем в два раза.

Принцип работы алгоритма: %пояснить за принцип работы%

/bin/bash: line 1: dc: command not found
